# 9장 · 문맥의 가치 — 조건부 엔트로피·상호정보량 🔵

웹 챕터 **"문맥의 가치"** 와 짝을 이루는 Python 실습입니다.
웹 벤다이어그램에서 잡은 감을, 여기서는 **실제 영어 텍스트에서 "앞 글자 힌트의 가치"를
직접 측정**하며 굳힙니다. 캡스톤(bigram 모델)의 예고편이기도 해요.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch9')

## 1. 결합분포에서 H(X), H(X|Y), I(X;Y) 뽑아내기

웹의 예보 예제(날씨 반반, 예보 90% 적중)를 결합분포 표에서 직접 계산합니다.

In [ ]:
import numpy as np

def entropy(p):
    p = np.asarray(p, dtype=float).ravel()
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

def cond_entropy(joint):
    """H(X|Y) — 열(Y)마다 조건부 분포의 H 를 p(y) 로 가중 평균. joint[x, y]."""
    p_y = joint.sum(axis=0)
    total = 0.0
    for j, py in enumerate(p_y):
        if py > 0:
            total += py * entropy(joint[:, j] / py)
    return total

def mutual_info(joint):
    """I(X;Y) = H(X) − H(X|Y)."""
    p_x = joint.sum(axis=1)
    return entropy(p_x) - cond_entropy(joint)

# 예보 예제: s = 예보 적중률. 대각선에 s/2 씩, 나머지에 (1-s)/2 씩.
def weather_joint(s):
    return np.array([[s / 2, (1 - s) / 2],
                     [(1 - s) / 2, s / 2]])

joint90 = weather_joint(0.9)
print('결합분포 (행 = 실제 날씨, 열 = 예보):')
print(joint90)
print(f'\nH(X)   = {entropy(joint90.sum(axis=1)):.4f} 비트  (예보 없이)')
print(f'H(X|Y) = {cond_entropy(joint90):.4f} 비트  (예보를 본 뒤, 3장의 0.469!)')
print(f'I(X;Y) = {mutual_info(joint90):.4f} 비트  (예보의 가치)')
assert abs(mutual_info(joint90) - (1 - entropy([0.9, 0.1]))) < 1e-9

## 2. 적중률과 정보량: U자 곡선의 반전

적중률 s 를 0부터 1까지 돌리며 I 를 그립니다. **양 끝(항상 맞음·항상 틀림)에서 만점,
한가운데(동전 던지기)에서 0**. 상호정보량은 "맞느냐"가 아니라 "얽혀 있느냐"를 잽니다.

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

ss = np.linspace(0.001, 0.999, 400)
Is = np.array([mutual_info(weather_joint(s)) for s in ss])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ss, Is, color='#7c3aed', lw=2.5)
for s, name in [(0.5, '동전 예보 I=0'), (0.9, '90% 적중 I=0.531')]:
    ax.plot(s, mutual_info(weather_joint(s)), 'o', color='#4f46e5', ms=8)
    ax.annotate(name, (s, mutual_info(weather_joint(s))), textcoords='offset points', xytext=(8, 8), fontsize=10)
ax.set_xlabel('예보 적중률 s')
ax.set_ylabel('I(X;Y): 예보의 가치 (비트)')
ax.set_title('항상 틀리는 예보(s=0)도 만점, 뒤집어 들으면 되니까')
ax.grid(True, color='#eef2f7')
plt.show()

## 3. 성질 검산: 대칭성, 그리고 KL 과의 동치

무작위 결합분포 1,000개로 두 가지를 검산합니다:
- **대칭**: I(X;Y) = I(Y;X)
- **5장 연결**: I = D(결합분포 ‖ 주변분포의 곱), 곧 "독립이라는 착각의 비용"

In [ ]:
def kl(p, q):
    p, q = np.asarray(p, dtype=float).ravel(), np.asarray(q, dtype=float).ravel()
    mask = p > 0
    return float(np.sum(p[mask] * np.log2(p[mask] / q[mask])))

rng = np.random.default_rng(42)
max_sym_gap, max_kl_gap = 0.0, 0.0
for _ in range(1000):
    j = rng.random((3, 4)) + 1e-6            # 3×4 무작위 결합분포
    j /= j.sum()
    i_xy = mutual_info(j)                     # I(X;Y)
    i_yx = mutual_info(j.T)                   # I(Y;X) — 행/열 뒤집기
    product = np.outer(j.sum(axis=1), j.sum(axis=0))   # 독립이라고 믿는 지도 p(x)p(y)
    d = kl(j, product)                        # D(결합 ‖ 곱)
    max_sym_gap = max(max_sym_gap, abs(i_xy - i_yx))
    max_kl_gap = max(max_kl_gap, abs(i_xy - d))

print(f'1000개 분포에서 |I(X;Y) − I(Y;X)| 최대: {max_sym_gap:.2e}  → 대칭 ✅')
print(f'1000개 분포에서 |I − D(결합‖곱)| 최대 : {max_kl_gap:.2e}  → KL 동치 ✅')
assert max_sym_gap < 1e-9 and max_kl_gap < 1e-9
print('\n상호정보량 = "둘이 독립이라고 믿는 지도"와의 KL. 얽힘의 크기 = 착각의 초과 비트.')

## 4. 실전: 진짜 텍스트에서 "앞 글자 힌트의 가치" 측정

2장부터 쓰던 그 영어 문장에서, **X = 다음 글자, Y = 바로 앞 글자**의 결합분포를 세어
문맥 한 글자의 가치를 실측합니다. LLM 이 문맥을 읽는 이유의 미니어처이자,
캡스톤 bigram 모델의 예고편!

In [ ]:
text = ('information theory studies the quantification storage and communication '
        'of information the field was established by claude shannon in the forties')

chars = sorted(set(text))
idx = {c: i for i, c in enumerate(chars)}
n = len(chars)

# 결합분포: joint[다음 글자, 앞 글자] — 연속한 글자쌍을 전부 센다
joint = np.zeros((n, n))
for prev, nxt in zip(text[:-1], text[1:]):
    joint[idx[nxt], idx[prev]] += 1
joint /= joint.sum()

H_X = entropy(joint.sum(axis=1))      # 힌트 없이: 다음 글자의 엔트로피
H_cond = cond_entropy(joint)          # 앞 글자를 알 때 남는 것
I = H_X - H_cond

print(f'H(다음 글자)            = {H_X:.3f} 비트  (≈ 2^{H_X:.2f} = {2**H_X:.1f}갈래 고민)')
print(f'H(다음 글자 | 앞 글자)  = {H_cond:.3f} 비트  (≈ {2**H_cond:.1f}갈래 고민)')
print(f'I = 앞 글자 힌트의 가치 = {I:.3f} 비트')
print(f'\n글자 하나를 미리 보는 것만으로 고민이 {2**H_X:.1f}갈래 → {2**H_cond:.1f}갈래로!')

# 구체적인 예: 'q' 다음과 't' 다음
for prev in ['q', 't']:
    col = joint[:, idx[prev]]
    if col.sum() > 0:
        cond = col / col.sum()
        top = np.argsort(cond)[::-1][:3]
        tops = ', '.join(f"'{chars[i]}' {cond[i]*100:.0f}%" for i in top if cond[i] > 0)
        print(f"앞 글자 '{prev}' 다음: H = {entropy(cond):.2f}비트, 상위: {tops}")
assert I > 0.5, '문맥의 가치가 너무 작다?'

'q' 다음은 사실상 'u' 확정(H ≈ 0), 't' 다음은 여전히 헷갈림. **문맥의 가치는 문맥마다
다르고**, 유용한 문맥에 집중하는 장치가 어텐션(4부)입니다.

## 5. 직접 풀어보기 ✏️

예보가 **100% 적중**(완전 동기)이라면 상호정보량 I(X;Y) 는 몇 비트일까요?
(날씨는 반반, H(X) = 1비트)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 완전 동기면 예보를 본 뒤 남는 불확실성 H(X|Y) = 0 — I = H(X) − H(X|Y)

answer = mutual_info(weather_joint(1.0))
print('내 답:', answer)
assert abs(answer - 1) < 1e-9, '다시 생각해 보세요 — 예보만 보면 날씨를 다 아는 상황'
print('✅ 잘했어요! I = 1비트 만점. 겹침이 원 전체를 덮은 상태입니다.')

---
**오늘의 한 줄**: H(X|Y) = 알고 난 뒤 남는 불확실성, I(X;Y) = 줄어든 만큼 = 힌트의 가치.
대칭이고, 독립이면 0, 항상 틀리는 힌트도 만점.

다음은 대단원인 🎓 **캡스톤**: 1부 신경망의 MSE 를 크로스 엔트로피로 갈아 끼우고,
문자 bigram 언어모델의 퍼플렉시티를 재고, temperature 로 문장을 생성합니다. 🙂